# 1. Project Title: Movie Review Sentiment Analysis (NLP)

**Kit:** AI Practical Project Learning Kit · **Project 3 of 5** · **Domain:** Natural Language Processing · **Difficulty:** Beginner

Run the cells from top to bottom. No GPU is needed.

## 2. Project Overview

Computers cannot read words, only numbers. In this project you learn the classic NLP recipe: **clean the text → turn words into numbers (Bag of Words, TF-IDF) → train a classifier (Logistic Regression) → measure it → use it on new sentences**. You will train on 25,000 real IMDb reviews and test on 25,000 different reviews.

## 3. Real-World Problem Statement

Studios, shops and apps receive thousands of reviews a day. Reading them all is impossible, but a company still wants to know *"are people happy?"*. **Sentiment analysis** labels each text as positive or negative automatically so a dashboard can track mood over time.

**Our task:** read a movie review and predict **Positive** or **Negative**.

## 4. Project Objectives

1. Explain what NLP is and why text must become numbers.
2. Do text preprocessing: lowercase, tokenize, remove stop-words.
3. Build **Bag of Words** and **TF-IDF** features.
4. Train **Logistic Regression** and compare it with Naive Bayes.
5. Evaluate with accuracy, **precision, recall, F1** and a confusion matrix.
6. Predict new reviews and see which words drove the decision.
7. Study correct and incorrect predictions and understand the model's limits.

## 5. What We Will Build

`Review text → cleaning → TF-IDF numbers → Logistic Regression → "Positive" / "Negative" + confidence`

## 6. Technologies and Libraries Used

| Library | Purpose |
|---|---|
| `datasets` (Hugging Face) | Downloads the IMDb dataset |
| `pandas`, `numpy` | Tables and numbers |
| `re` | Pattern-based text cleaning (regular expressions) |
| `scikit-learn` | Vectorizers, models and metrics |
| `matplotlib`, `seaborn` | Charts |
| `joblib` | Saving the trained pipeline |

## 7. System Architecture

```
IMDb reviews (25k train / 25k test)
      │
      ▼
 Clean text (lowercase, strip HTML, keep letters)
      │
      ▼
 Vectorize: Bag of Words  or  TF-IDF  (+ stop-word removal, tokenization)
      │
      ▼
 Logistic Regression (and Naive Bayes for comparison)
      │
      ▼
 Evaluate: accuracy · precision · recall · F1 · confusion matrix
      │
      ▼
 Predict new text ─► Save pipeline
```

## 8. Understanding the Dataset

The **IMDb Large Movie Review Dataset** (Maas et al., 2011) has **50,000 labelled reviews**: 25,000 for training and 25,000 for testing, each half positive and half negative. Labels: `0 = negative` (rating ≤ 4 out of 10) and `1 = positive` (rating ≥ 7). Neutral reviews were left out. Because the classes are balanced, plain **accuracy** is a fair first metric.

Reviews are in English and are written by general users, so the model learns the style of *movie reviews*; it may be unreliable on tweets, product reviews or other languages.

## 9. Dataset Download / Loading

In [ ]:
# Section 9 - imports and loading (first run downloads ~80 MB)
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from datasets import load_dataset

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report, ConfusionMatrixDisplay)

SEED = 42
np.random.seed(SEED)
sns.set_theme(style="whitegrid")

imdb = load_dataset("stanfordnlp/imdb")
train_df = imdb["train"].to_pandas()
test_df = imdb["test"].to_pandas()
print("Train:", train_df.shape, "| Test:", test_df.shape)

## 10. Dataset Exploration

In [ ]:
# Section 10a - look at samples and the label balance
LABELS = {0: "Negative", 1: "Positive"}
print(train_df["label"].map(LABELS).value_counts(), "\n")
for i in [0, 12500]:
    print(LABELS[train_df.loc[i, "label"]], "->", train_df.loc[i, "text"][:300], "...\n")

In [ ]:
# Section 10b - review length (in words) by sentiment
train_df["n_words"] = train_df["text"].str.split().str.len()
print(train_df.groupby("label")["n_words"].describe().round(1))

plt.figure(figsize=(9, 4))
sns.histplot(data=train_df.assign(Sentiment=train_df["label"].map(LABELS)), x="n_words",
             hue="Sentiment", bins=60, element="step", binrange=(0, 1000))
plt.title("Review length (words)"); plt.show()

**Why this matters:** the two classes have similar length distributions, so length alone cannot tell sentiment, and the model must learn from the *words*. Reviews vary from a few dozen to over a thousand words.

## 11. Data Preprocessing

Raw reviews contain HTML tags (`<br />`), capital letters, punctuation and numbers. Our pipeline:

1. **Lowercase** so "Great" and "great" are the same word.
2. **Remove HTML tags** such as `<br />`.
3. **Keep only letters** (and apostrophes) so punctuation does not become noise.
4. **Tokenization**: split text into word tokens (the vectorizer does this for us).
5. **Stop-word removal**: drop very common words ("the", "is", "and") that carry little sentiment.

We keep the word **"not"**. It is on scikit-learn's stop-word list, yet "not good" and "good" mean opposite things. Removing it would hurt the model.

In [ ]:
# Section 11 - cleaning function and a custom stop-word list
def clean_text(text):
    text = text.lower()
    text = re.sub(r"<[^>]+>", " ", text)          # remove HTML tags
    text = re.sub(r"[^a-z' ]", " ", text)          # keep letters, apostrophes and spaces
    text = re.sub(r"\s+", " ", text).strip()      # collapse repeated spaces
    return text

NEGATIONS = {"not", "no", "nor", "never", "n't", "cannot"}
STOP_WORDS = sorted(set(ENGLISH_STOP_WORDS) - NEGATIONS)       # keep negation words

example = "This movie was <br /><br />GREAT!!! I loved it, 10/10 -- but NOT the ending."
print("Before:", example)
print("After :", clean_text(example))
print("Tokens:", [w for w in clean_text(example).split() if w not in STOP_WORDS])

train_df["clean"] = train_df["text"].apply(clean_text)
test_df["clean"] = test_df["text"].apply(clean_text)
X_train, y_train = train_df["clean"], train_df["label"]
X_test, y_test = test_df["clean"], test_df["label"]
print("\nCleaned", len(train_df) + len(test_df), "reviews")

## 12. Understanding the Algorithm

### Bag of Words (BoW)
Make a vocabulary of all words, then describe each review by **how many times each word appears**. Word order is thrown away, like emptying a bag of words onto a table.

### TF-IDF
BoW over-counts words that appear everywhere. **TF-IDF** = *Term Frequency × Inverse Document Frequency*: a word scores high if it is frequent **in this review** but **rare across all reviews** (like "masterpiece"), and low if it is everywhere (like "movie"). `ngram_range=(1, 2)` also uses two-word phrases ("not good"), `min_df=5` ignores words seen in fewer than 5 reviews, and `max_features` caps the vocabulary size.

### Logistic Regression
Despite the name, it is a **classifier**. It gives each word a weight (positive words get positive weights), adds them up for a review, and squashes the total into a probability between 0 and 1 with the **sigmoid** function. Probability above 0.5 → Positive. The weights are also easy to inspect, which makes the model explainable.

### Naive Bayes (comparison)
A very fast model that treats words as independent clues and multiplies their probabilities. We use it as a baseline.

## 13. Building the Model

A scikit-learn **Pipeline** chains *vectorizer → classifier* into a single object that goes from raw text to a prediction, so training and prediction always use the same steps. We build three candidates to compare.

In [ ]:
# Section 13 - three candidate pipelines
tfidf_args = dict(stop_words=STOP_WORDS, ngram_range=(1, 2), min_df=5, max_features=100000, sublinear_tf=True)

models = {
    "BoW + Logistic Regression": make_pipeline(
        CountVectorizer(stop_words=STOP_WORDS, min_df=5, max_features=50000),
        LogisticRegression(max_iter=1000, random_state=SEED)),
    "TF-IDF + Logistic Regression": make_pipeline(
        TfidfVectorizer(**tfidf_args),
        LogisticRegression(C=4.0, max_iter=1000, random_state=SEED)),
    "TF-IDF + Naive Bayes": make_pipeline(
        TfidfVectorizer(**tfidf_args),
        MultinomialNB()),
}
print(list(models))

## 14. Training the Model

Each pipeline learns its vocabulary and weights from the 25,000 training reviews only. This can take a minute or two.

In [ ]:
# Section 14 - training
import time
for name, mdl in models.items():
    t0 = time.time()
    mdl.fit(X_train, y_train)
    print(f"{name:32s} trained in {time.time() - t0:5.1f}s")

## 15. Model Evaluation

We predict the 25,000 held-out test reviews. For the *Positive* class:

* **Precision** = of the reviews we called positive, how many really were?
* **Recall** = of the truly positive reviews, how many did we find?
* **F1-score** = the balance (harmonic mean) of the two.
* **Confusion matrix** shows the four outcomes: true/false positives and negatives.

In [ ]:
# Section 15 - evaluation on the TEST set
rows, preds = [], {}
for name, mdl in models.items():
    p = mdl.predict(X_test); preds[name] = p
    rows.append({"Model": name,
                 "Accuracy": accuracy_score(y_test, p),
                 "Precision": precision_score(y_test, p),
                 "Recall": recall_score(y_test, p),
                 "F1": f1_score(y_test, p)})
results = pd.DataFrame(rows).set_index("Model")
display(results.round(4))

best_name = results["F1"].idxmax()
best_model = models[best_name]
print("\nBest by F1:", best_name)
print(classification_report(y_test, preds[best_name], target_names=["Negative", "Positive"], digits=3))

**Reading the result:** accuracy is the share of all 25,000 reviews labelled correctly. Because the classes are balanced, precision and recall for each class should be close to each other. The numbers above are from your run; they are not hard-coded in this text.

## 16. Data Visualization

In [ ]:
# Section 16a - confusion matrix of the best model and model comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cm = confusion_matrix(y_test, preds[best_name])
ConfusionMatrixDisplay(cm, display_labels=["Negative", "Positive"]).plot(ax=axes[0], cmap="Blues", colorbar=False)
axes[0].set_title("Confusion matrix: " + best_name)
results.plot(kind="bar", ax=axes[1], rot=15, ylim=(0.6, 1.0))
axes[1].set_title("Model comparison (higher is better)")
plt.tight_layout(); plt.show()

The diagonal of the confusion matrix holds correct predictions. Off-diagonal cells are the two kinds of mistake: *false positives* (negative review called positive) and *false negatives*.

In [ ]:
# Section 16b - which words drive the decision? (Logistic Regression weights)
lr_name = "TF-IDF + Logistic Regression"
vec, clf = models[lr_name].steps[0][1], models[lr_name].steps[1][1]
words = np.array(vec.get_feature_names_out())
order = np.argsort(clf.coef_[0])
top_neg, top_pos = words[order[:15]], words[order[-15:]]
coef = clf.coef_[0]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].barh(top_neg, coef[order[:15]], color="tomato"); axes[0].set_title("Strongest NEGATIVE words")
axes[1].barh(top_pos, coef[order[-15:]], color="seagreen"); axes[1].set_title("Strongest POSITIVE words")
plt.tight_layout(); plt.show()

These are the words the model trusts most. Check whether they make sense to you, since this is how you catch a model that learned something silly.

## 17. Testing with New Inputs

For probability models we show **confidence** = the predicted probability of the chosen class. For a new sentence we also list the words that pushed the decision.

In [ ]:
# Section 17a - predict new reviews
def predict_review(text, model=models[lr_name]):
    if not isinstance(text, str) or len(text.strip()) < 3:
        return "Please enter a longer review."
    cleaned = clean_text(text)
    if not cleaned:
        return "No readable English letters found in the text."
    proba = model.predict_proba([cleaned])[0]
    label = int(np.argmax(proba))
    return f"{LABELS[label]} (confidence {proba[label]:.1%})"

tests = [
    "An absolute masterpiece. Brilliant acting and a story that stays with you.",
    "Boring, predictable and far too long. A complete waste of my time.",
    "The film was not good.",
    "Not bad at all, I actually enjoyed it.",
    "It was fine I guess.",
]
for t in tests:
    print(f"{t!r:80s} -> {predict_review(t)}")

Try the *tricky* sentences: negation (`not good`, `not bad`), mild opinions and sarcasm are the hardest for a bag-of-words model. Notice how the confidence drops for ambiguous text.

In [ ]:
# Section 17b - interactive: type your own review
try:
    user_text = input("Type a movie review (or press Enter to skip): ")
except EOFError:
    user_text = ""
if user_text.strip():
    print(predict_review(user_text))
else:
    print("Skipped. Re-run this cell and type a review.")

In [ ]:
# Section 17c - examples of correct and incorrect predictions on the test set
p = preds[lr_name]
proba = models[lr_name].predict_proba(X_test)
test_df["pred"] = p
test_df["conf"] = proba.max(axis=1)

def show(df, title, n=3):
    print("=" * 70, "\n" + title)
    for _, r in df.head(n).iterrows():
        print(f"actual={LABELS[r['label']]:8s} predicted={LABELS[r['pred']]:8s} conf={r['conf']:.0%}")
        print("   ", r["text"][:220].replace("<br />", " "), "...\n")

right = test_df[test_df["pred"] == test_df["label"]].sort_values("conf", ascending=False)
wrong = test_df[test_df["pred"] != test_df["label"]].sort_values("conf", ascending=False)
show(right, "MOST CONFIDENT CORRECT predictions")
show(wrong, "MOST CONFIDENT MISTAKES (look for sarcasm, mixed opinions, mislabelled data)")

Reading the most confident mistakes is the best way to understand a model's blind spots: sarcasm, reviews that praise the acting but hate the plot, and plain labelling noise.

## 18. Saving the Trained Model

In [ ]:
# Section 18 - save and reload the whole pipeline (vectorizer + classifier)
joblib.dump(models[lr_name], "sentiment_pipeline.joblib", compress=3)
reloaded = joblib.load("sentiment_pipeline.joblib")
print("Reloaded model says:", reloaded.predict([clean_text("what a wonderful, moving film")])[0], "(1 = Positive)")
try:
    from google.colab import files
    files.download("sentiment_pipeline.joblib")
except ImportError:
    print("(Not running in Colab - the file is in the current folder.)")

## 19. Final Output

In [ ]:
# Section 19 - final summary
print("=" * 60); print("SENTIMENT ANALYSIS - FINAL SUMMARY"); print("=" * 60)
display(results.round(4))
print(f"Best model by F1 : {best_name}")
print("Demo: ", predict_review("A heartfelt story with wonderful performances."))

## 20. How the Complete Project Works

1. Download 50,000 labelled IMDb reviews.
2. Clean them (lowercase, remove HTML, keep letters) and drop stop-words except negations.
3. Convert text to numbers with Bag of Words or TF-IDF (including two-word phrases).
4. Train Logistic Regression and Naive Bayes on the training half.
5. Evaluate on the untouched test half using accuracy, precision, recall, F1 and a confusion matrix.
6. Inspect the strongest words and the most confident mistakes.
7. Wrap everything in one pipeline, save it, and use it for new text.

## 21. Limitations

* **Word order is lost**, so negation, sarcasm and phrases like "not what I expected" are hard.
* Trained on **movie reviews in English**; accuracy falls on other topics or languages.
* Only two classes: no neutral or mixed sentiment.
* The model may reflect biases in the reviews it learned from.
* Confidence is a model score, not a guarantee of being right.

## 22. Real-World Applications

Brand monitoring on social media · customer-support triage · app-store and product-review dashboards · market research · early detection of unhappy customers.

## 23. Future Enhancements

* Use a pretrained transformer (for example DistilBERT) via Hugging Face `pipeline("sentiment-analysis")` and compare it with this model.
* Add a neutral class and use a 5-star dataset.
* Tune `C`, `ngram_range` and `max_features` with `GridSearchCV`.
* Use word embeddings or LSTM networks to keep word order.
* Serve the saved pipeline from a Gradio or Streamlit app.

## 24. Student Mini Challenges

1. Add your own stop-words (such as "movie", "film") and see whether the scores change.
2. Change `ngram_range` to `(1, 1)` and then `(1, 3)`. What happens to accuracy and training time?
3. Print the 20 words with the largest weights in the Naive Bayes model (`feature_log_prob_`).
4. Find five test reviews the model gets wrong and write why you think it failed.
5. Lower `max_features` to 5,000. How much accuracy do you lose?
6. Add a sentence of your own that contains sarcasm and report what the model predicts.